In [4]:
import os, rasterio, sys, rioxarray, pyflwdir, shutil, dotenv, glob, gc
sys.path.append('backend/app/')
from rasterio.io import MemoryFile
from rasterio.features import rasterize
from rasterio.merge import merge
import geopandas as gpd, pandas as pd
import numpy as np, xarray as xr
from backend.app.services import flow_functions, functions
from shapely.geometry import Polygon
from shapely import force_2d
from pyflwdir import dem
from hydromt_wflow import WflowSbmModel
from tqdm import tqdm
from owslib.wcs import WebCoverageService
from whitebox.whitebox_tools import WhiteboxTools
from terracatalogueclient import Catalogue
dotenv.load_dotenv()
wtb = WhiteboxTools()
wtb.set_verbose_mode(False)
np.random.seed(42)

In [ ]:
ESA_USERNAME=vanlnntnu@gmail.com
ESA_PASSWORD=@19102017HanThuLam@
CORINE_TOKEN_ID=28c777a4ad45603f0d5c6d3600127c7de855496f
CORINE_TOKEN_TITLE=CORINE_2018_Platform

## Child Functions

In [2]:
def create_forcing(time, ny, nx, values, single_value=True):
    if single_value:
        data = np.empty((len(time), ny, nx), dtype=np.float32)
        data[:] = values[:, None, None]
    

    return data

# def assign_nodata(path, nodata):
#     with rasterio.open(path, "r+") as src:
#         arr = src.read(1)
#         arr = np.where(np.isnan(arr), nodata, arr)
#         src.write(arr, 1)
#         src.nodata = nodata

In [2]:
sample_folder, test_folder = 'inputs', 'test'
catchment_path = os.path.join(sample_folder, 'catchment.geojson')
terrain_path = os.path.join(sample_folder, 'dtm10.tif')
lake_path = os.path.join(sample_folder, 'BrusdalLake.geojson')
river_path = os.path.join(sample_folder, 'river.geojson')
# land_path = os.path.join(sample_folder, 'land.geojson')
terrain = rioxarray.open_rasterio(terrain_path).squeeze()
catchment = gpd.read_file(catchment_path)
initial_param = [0.25,0.3,50,0,0,500,0,0,100]
# land = gpd.read_file(land_path)
NODATA_FLOAT, NODATA_INT = -9999.0, 0
raw_dir = os.path.normpath(os.path.join(f'{test_folder}/data/raw'))
if not os.path.exists(raw_dir): os.makedirs(raw_dir)
raw_path = os.path.normpath(os.path.join(raw_dir, "dtm_raw.tif"))
lake = gpd.read_file(lake_path).to_crs(terrain.rio.crs)

## Create a raw terrain that is clipped to catchment

In [ ]:
# Clip terrain to catchment
catchment_UTM = catchment.to_crs(terrain.rio.crs)
temp = catchment_UTM.copy()
temp['geometry'] = temp['geometry'].buffer(10)
terrain_clipped = flow_functions.clip_catchment(temp, terrain)
terrain_clipped.rio.to_raster(raw_path)

## Process river data

In [67]:
# Create river from DEM if it doesn't exist
if not os.path.exists(river_path):
    threshold, min_length = 0.1, 100
    with rasterio.open(raw_path) as src:
        dem_array = src.read(1).astype(np.float32)
        transform, profile = src.transform, src.profile
        crs, nodata = src.crs, src.nodata
    mask = np.isnan(dem_array) | (dem_array == nodata)
    filled_array, flwdir_array = dem.fill_depressions(elevtn=dem_array, max_depth=-1)
    flwdir_array = np.where(mask, NODATA_INT, flwdir_array)
    flw = pyflwdir.from_dem(filled_array, transform=transform, latlon=crs.is_geographic)
    uparea = flw.upstream_area(unit="km2")
    river_mask = uparea > threshold
    features = flw.streams(river_mask)
    gdf = gpd.GeoDataFrame.from_features(features, crs=crs)
    lake['geometry'] = lake['geometry'].buffer(10)
    # Clipp river to lake
    clipped_river = gdf.overlay(lake, how='difference')
    clipped_river['lenght'] = clipped_river['geometry'].length
    clipped_river = clipped_river[clipped_river['lenght'] > min_length]
    clipped_river.reset_index(drop=True, inplace=True)
    clipped_river = clipped_river[['geometry']]
    river = clipped_river.reindex(columns=['rivwth', 'rivdph', 'geometry'])
else:
    river = gpd.read_file(river_path)
    river = river.rename(columns={'width': 'rivwth', 'depth': 'rivdph'})
    river = river[['rivwth', 'rivdph', 'geometry']]
# Create a random value for each river
cols = {'rivwth': (0.05, 2), 'rivdph': (1, 5)}
river_cols = river.columns.drop('geometry', errors='ignore')
for col in river_cols:
    river[col] = pd.to_numeric(river[col], errors='coerce')
for col, (low, high) in cols.items():
    river_mask = river[col].isna() | (river[col] == 'None')
    river.loc[river_mask, col] = np.round(np.random.uniform(low, high, river_mask.sum()), 3)
river = river[river.is_valid].reset_index(drop=True)
# Process river
river["geometry"] = river.geometry.apply(lambda g: force_2d(g))
# Write file
river.to_file(os.path.normpath(os.path.join(f'{test_folder}/data/river', 'river.gpkg')), driver='GPKG')

## Create template hydro data

In [64]:
# Prepare template raster dataset
with rasterio.open(raw_path) as src:
    dem_array = src.read(1).astype(np.float32)
    profile, transform, crs = src.profile, src.transform, src.crs
hydro_dir = os.path.normpath(f'{test_folder}/data/hydro')
if os.path.exists(hydro_dir): shutil.rmtree(hydro_dir)
os.makedirs(hydro_dir)
NODATA_DEM, NODATA_INT = -9999.0, 0
with rasterio.open(raw_path) as src:
    dem_array = src.read(1).astype(np.float32)
    transform, profile = src.transform, src.profile
    crs, nodata = src.crs, src.nodata
# Fill depressions
filled_array, flwdir_array = dem.fill_depressions(elevtn=dem_array, max_depth=-1)
flwdir_array = np.where(filled_array == NODATA_DEM, NODATA_INT, flwdir_array)
# Create slope
dx, dy = transform.a, abs(transform.e)
# Gradient elevation
gradient_array = np.where(filled_array == NODATA_DEM, np.nan, filled_array)
gy, gx = np.gradient(gradient_array, dy, dx)
slope_array = np.sqrt(gx**2 + gy**2)
slope_array = np.where(filled_array == NODATA_DEM, NODATA_DEM, slope_array)
# Create basins
flw = pyflwdir.from_dem(filled_array, transform=transform, latlon=crs.is_geographic)
basins_array = flw.basins()
basins_array = np.where(filled_array == NODATA_DEM, NODATA_DEM, basins_array)
# Create stream order
uparea_array = flw.upstream_area(unit='km2')
uparea_array = np.where(filled_array == NODATA_DEM, NODATA_DEM, uparea_array)
# Create stream mask and stream order
stream_mask = uparea_array > 0
strord_array = flw.stream_order(type='strahler', mask=stream_mask)
strord_array = np.where(filled_array == NODATA_DEM, NODATA_DEM, strord_array)
# Create upstream grid
upgrid_array = flw.upstream_area(unit='cell')
upgrid_array = np.where(filled_array == NODATA_DEM, NODATA_DEM, upgrid_array)
# Create river width
river = gpd.read_file(os.path.normpath(os.path.join(f'{test_folder}/data/river', 'river.gpkg')))
shapes = ((geom, value) for geom, value in zip(river.geometry, river["rivwth"]))
rivwth_array = rasterize(
    shapes=shapes, out_shape=(src.height, src.width),
    transform=transform, fill=NODATA_DEM, dtype="float32"
)
rivwth_array = np.where(filled_array == NODATA_DEM, NODATA_DEM, rivwth_array)
flow_functions.write_geotiff(filled_array, profile, os.path.join(hydro_dir, 'elevtn.tif'))
profile_flw = {**profile, 'dtype': np.uint8, 'nodata': NODATA_INT}
flow_functions.write_geotiff(flwdir_array, profile_flw, os.path.join(hydro_dir, 'flwdir.tif'))
flow_functions.write_geotiff(slope_array, profile, os.path.join(hydro_dir, 'lndslp.tif'))
flow_functions.write_geotiff(basins_array, profile, os.path.join(hydro_dir, 'basins.tif'))
flow_functions.write_geotiff(uparea_array, profile, os.path.join(hydro_dir, 'uparea.tif'))
flow_functions.write_geotiff(strord_array, profile, os.path.join(hydro_dir, 'strord.tif'))
flow_functions.write_geotiff(upgrid_array, profile, os.path.join(hydro_dir, 'upgrid.tif'))
flow_functions.write_geotiff(rivwth_array, profile, os.path.join(hydro_dir, 'rivwth.tif'))

## Prepare forcing data from the customized area

In [ ]:
# Read weather data
weather_path = os.path.join(sample_folder, 'alesund_weather.csv')
weather = pd.read_csv(weather_path, parse_dates=['datetime'], index_col='datetime')
weather_new = weather.loc['2025-01-01 00:00:00':'2025-01-10 00:00:00']

In [ ]:
# Create forcing nc file
time, crs = weather_new.index.to_numpy(), terrain.rio.crs
if crs is None: raise ValueError("Terrain has no crs")
ny, nx = terrain.rio.height, terrain.rio.width
forcing = {
    'precip': ['precip_mm', '(mm/h)'], 'temp': ['temp_C', '(degC)'],
    'kin': ['shortwave_Wm2', '(W/m^2)'], 'kout': ['longwave_Wm2', '(W/m^2)'],
    'wind': ['wind_mps', '(m/s)'], 'press_msl': ['pressure', '(Pa)']
}
forcing_dir = os.path.join(test_folder, 'data/forcing')
if not os.path.exists(forcing_dir): os.makedirs(forcing_dir)
out_path, datasets = os.path.join(forcing_dir, "my_forcing.nc"), {}
for item, values in forcing.items():
    data = weather_new[values[0]].values
    data_3d = create_forcing(time, ny, nx, data)
    datasets[item] = (('time', 'y', 'x'), data_3d, {'units': values[1]})
ds_final = xr.Dataset(
    data_vars=datasets, coords={"time": time, "y": terrain.y, "x": terrain.x}
)
ds_final.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
ds_final.rio.write_crs(crs, inplace=True)
encoding = {
    var: {"zlib": True, "complevel": 4, "shuffle": True, "chunksizes": (1, 256, 256)}
    for var in ds_final.data_vars
}
ds_final.to_netcdf(out_path, engine='netcdf4', encoding=encoding)

## Process soil data

In [ ]:
# Initialize variables
soil_types = {
    'clay': 'clyppt', 'sand': 'sndppt', 'silt': 'sltppt', 
    'bdod': 'bd', 'soc': 'oc', 'phh2o': 'ph'
}
depths = {
    '0-5cm_mean': 'sl1', '5-15cm_mean': 'sl2', '15-30cm_mean': 'sl3',
    '30-60cm_mean': 'sl4', '60-100cm_mean': 'sl5', '100-200cm_mean': 'sl6'
}
soil_dir = os.path.join(f'{test_folder}/data/soil')
if not os.path.exists(soil_dir): os.makedirs(soil_dir)
min_lon, min_lat, max_lon, max_lat = catchment.total_bounds
bbox = (float(min_lon), float(min_lat), float(max_lon), float(max_lat))

In [12]:
# Create soil thickness
with rasterio.open(terrain_path) as src:
    meta = src.meta.copy()
meta.update({"dtype": "float32", "nodata": -9999.0})
data = np.ones((meta["height"], meta["width"]), dtype="float32") * 100
data[data == meta["nodata"]] = 100
with rasterio.open(os.path.join(soil_dir, 'soilthickness.tif'), "w", **meta) as dst:
    dst.write(data, 1)

In [9]:
# Download soil data from ISRIC: https://files.isric.org/soilgrids/latest/data/
for item, name in tqdm(soil_types.items(), total=len(soil_types), desc='Downloading soil data'):
    wcs = WebCoverageService(f'https://maps.isric.org/mapserv?map=/map/{item}.map', version='1.0.0')
    for type, value in depths.items():
        idx = f'{item}_{type}'
        response = wcs.getCoverage(
            identifier=idx, crs='EPSG:4326', bbox=bbox,
            format='image/tiff', resx=0.0025, resy=0.0025
        )
        with MemoryFile(response.read()) as memfile:
            with memfile.open() as src:
                data = rioxarray.open_rasterio(src, masked=True)
                data_reprojected = data.rio.reproject_match(terrain)
            data_reprojected.rio.to_raster(os.path.join(soil_dir, f'{name}_{value}.tif'))

## Process land cover

In [ ]:
# Get data from ESA worldcover
user_name, password = os.getenv('ESA_USERNAME'), os.getenv('ESA_PASSWORD')
land_dir = os.path.normpath(os.path.join(test_folder, 'data/landcover'))
if not os.path.exists(land_dir): os.makedirs(land_dir)
catalogue = Catalogue().authenticate_non_interactive(user_name, password)
area = catchment.copy()
if area.crs != 'EPSG:4326': area = area.to_crs('EPSG:4326')
minx, miny, maxx, maxy = area.total_bounds
bbox = Polygon.from_bounds(minx, miny, maxx, maxy)
download_dir = os.path.join(land_dir, 'downloads')
if os.path.exists(download_dir): shutil.rmtree(download_dir)
# # Get name of landcover layer
# collections = catalogue.get_collections()
layers = [
    # 'urn:eop:VITO:ESA_WorldCover_10m_2020_V1', 
    'urn:eop:VITO:ESA_WorldCover_10m_2021_V2'
]
# Search for products in the WorldCover collection
product = catalogue.get_products(layers, geometry=bbox)
catalogue.download_products(product, download_dir, force=True)
pattern = os.path.join(download_dir, "**", "*_Map.tif")
files = glob.glob(pattern, recursive=True)
if len(files) == 0: raise ValueError("No *_Map.tif files found in directory")
# print(f"Found {len(files)} tiles")
if len(files) == 1:
    src_files = [rasterio.open(files[0])]
    mosaic, transform = src_files[0].read(), src_files[0].transform
elif len(files) > 1:
    src_files = [rasterio.open(f) for f in files]
    # Merge (mosaic)
    mosaic, transform = merge(src_files)
# Copy metadata
out_meta = src_files[0].meta.copy()
out_meta.update({
    "height": mosaic.shape[1], "width": mosaic.shape[2],
    "transform": transform, "compress": "lzw"
})
merge_path = os.path.join(land_dir, 'merged.tif')
with rasterio.open(merge_path, "w", **out_meta) as dest:
    dest.write(mosaic)
# Close files
for src in src_files: src.close()
shutil.rmtree(download_dir)
# Clip raster to catchment
ds = rioxarray.open_rasterio(merge_path).squeeze()
land_temp = ds.squeeze()
area['geometry'] = area['geometry'].buffer(0.001)
land_clip = flow_functions.clip_catchment(area, land_temp, land_temp.rio.nodata)
path = os.path.join(land_dir, 'esa_worldcover.tif')
land_clip = land_clip.rio.reproject(terrain.rio.crs)
land_clip.rio.to_raster(path)
del land_temp, land_clip, ds
gc.collect()
functions.safe_remove(merge_path)

In [ ]:
# Get data from CORINE 2018

In [7]:
# Run HydroMT
model_path = os.path.normpath(f'{test_folder}/model')
if os.path.exists(model_path): shutil.rmtree(model_path)
!hydromt build wflow_sbm "./test/model" -i "./test/build.yml" -d "./test/config.yml" -v

2026-05-21 18:14:02,682 - hydromt - log - INFO - HydroMT version: 1.3.1
2026-05-21 18:14:02,803 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from ./test/config.yml
2026-05-21 18:14:02,843 - hydromt.model.model - model - INFO - Initializing wflow_sbm model from hydromt_wflow (v1.0.1).
2026-05-21 18:14:02,843 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from C:\Envs\hyd_ai\Lib\site-packages\hydromt_wflow\data\parameters_data.yml
2026-05-21 18:14:02,869 - hydromt.hydromt_wflow.wflow_base - wflow_base - INFO - Supported Wflow.jl version v1+
2026-05-21 18:14:02,871 - hydromt.hydromt_wflow.components.config - config - INFO - Reading default config file from C:/Envs/hyd_ai/Lib/site-packages/hydromt_wflow/data/wflow_sbm/wflow_sbm.toml.
2026-05-21 18:14:02,875 - hydromt - log - INFO - HydroMT version: 1.3.1
2026-05-21 18:14:02,875 - hydromt.model.model - model - INFO - build: setup_config
2026-05-21 18:14:02,876 - hydromt.m